# PMIL reproduction — Google Colab runbook

This notebook runs the independent, attributed PMIL reproduction. It does not claim authorship of the PMIL method. Use only Camelyon16 data you downloaded under the official dataset terms. Results stay blank until your experiment finishes.

**Before starting:** In Colab select **Runtime → Change runtime type → T4 GPU**. Keep the dataset and generated features in Google Drive, not in temporary `/content` storage.

## 1. Upload the project ZIP

Upload the project zip file you downloaded (e.g. `pmil-colab-reproduction-v5.zip`) when the picker opens. The cell below picks up whichever `.zip` you select, so the exact version number doesn't matter.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose the zip you uploaded
zip_name = next(name for name in uploaded if name.endswith('.zip'))
!mkdir -p /content/pmil-upload
!unzip -q "$zip_name" -d /content/pmil-upload
%cd /content/pmil-upload/pmil-colab-reproduction


## 2. Install the runtime

This installs the current CPU/GPU-compatible PyTorch dependencies and OpenSlide support for raw whole-slide images.

In [ ]:
!apt-get -qq update && apt-get -qq install -y libopenslide0
!pip -q install -r requirements.txt -e .
!python -c "import torch, pmil; print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.is_available())"

## 3. Mount Google Drive

Create these folders in Drive before continuing:

- `MyDrive/camelyon16/training/` — your downloaded Camelyon16 WSIs
- `MyDrive/camelyon16/train_labels.csv` — permitted slide-level labels with columns `slide_id,label`
- `MyDrive/pmil/features/` — generated feature bags and checkpoints

Download the dataset yourself from the [official Camelyon challenge data page](https://camelyon17.grand-challenge.org/Data/). Do not upload WSIs or patient data to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WSI_DIR = '/content/drive/MyDrive/camelyon16/training'
LABELS_CSV = '/content/drive/MyDrive/camelyon16/train_labels.csv'
DRIVE_OUT = '/content/drive/MyDrive/pmil'
!mkdir -p "$DRIVE_OUT/features/train" "$DRIVE_OUT/features/val" "$DRIVE_OUT/artifacts"


## 3b. Generate `train_labels.csv` (skip if you already have one)

If you haven't already created `MyDrive/camelyon16/train_labels.csv`, generate it from the standard Camelyon16 training folder names (which contain `normal` or `tumor`). Skip this cell if you already have a correct `slide_id,label` CSV — running it will overwrite `LABELS_CSV`.

In [ ]:
!python -m pmil.make_camelyon16_labels \
  --wsi-dir "$WSI_DIR" \
  --output "$LABELS_CSV"


## 4. Build coordinate-only train/validation manifests

This reads each WSI sequentially and saves coordinates—not extracted patch images. Start with `--level 1` only as a quick pipeline check. For a method-faithful experiment, set the level and tiling parameters to the resolution/protocol you document; the original paper reports 40× evaluation. This step can take a long time on full slides.

Your labels CSV must have at least two examples in each class for a stratified split.

In [ ]:
!python -m pmil.prepare_camelyon16 \
  --wsi-dir "$WSI_DIR" \
  --labels-csv "$LABELS_CSV" \
  --output-dir data/splits \
  --level 1 --tile-size 256 --stride 256 \
  --min-tissue-fraction 0.05 --seed 24


## 5. Train Max-MIL and save training features

Patches are streamed one slide and batch at a time. The model keeps only the current batch and selected top instances in memory.

In [ ]:
!python -m pmil.train_maxmil \
  --slides-csv data/splits/train.csv --val-csv data/splits/val.csv \
  --patch-root / --checkpoint "$DRIVE_OUT/artifacts/maxmil.pt" \
  --feature-dir "$DRIVE_OUT/features/train" \
  --epochs 30 --batch-size 128 --top-k 1 --top-n-features 1000 --seed 24


## 6. Extract validation features, discover prototypes, and train PMIL

Run these in order. Do not report metrics until the final run completes and you have recorded your own split and settings.

In [ ]:
!python -m pmil.extract_features \
  --slides-csv data/splits/val.csv --patch-root / \
  --checkpoint "$DRIVE_OUT/artifacts/maxmil.pt" \
  --feature-dir "$DRIVE_OUT/features/val" \
  --top-n-features 1000 --batch-size 128 --seed 24

!python -m pmil.cluster_prototypes \
  --slides-csv data/splits/train.csv \
  --feature-dir "$DRIVE_OUT/features/train" \
  --output "$DRIVE_OUT/artifacts/prototypes.npy" --seed 24

!python -m pmil.train_pmil \
  --train-csv data/splits/train.csv --val-csv data/splits/val.csv \
  --train-features "$DRIVE_OUT/features/train" \
  --val-features "$DRIVE_OUT/features/val" \
  --prototypes "$DRIVE_OUT/artifacts/prototypes.npy" \
  --instance-checkpoint "$DRIVE_OUT/artifacts/maxmil.pt" \
  --checkpoint "$DRIVE_OUT/artifacts/pmil.pt" --epochs 30 --seed 24


## 7. After your run

Save your genuine command, random seed, train/validation split, hardware, training logs, and held-out metrics. Add them to `results/` only after you verify them. Keep the attribution and citation files unchanged when uploading the repository to GitHub.